In [7]:
# ======================================================================
# Regression (Premium Prediction):
# ======================================================================


# ----------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import joblib

# ----------------------------------------------------------------------
# Absolute base directories 
# ----------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project")
INSUREAI_ROOT = PROJECT_ROOT / "InsureAI"

PROCESSED_DIR = INSUREAI_ROOT / "data" / "processed"
REPORTS_DIR   = PROJECT_ROOT / "reports"
MODELS_DIR    = PROJECT_ROOT / "models"

# Ensure output folders exist
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset
df_insurance = pd.read_csv(r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\data\raw\insurance.csv")

def main():
    # --------------------------------------------------------------
    # 1️⃣ Load cleaned premium CSV
    # --------------------------------------------------------------
    csv_path = PROCESSED_DIR / "insurance_cleaned.csv"
    if not csv_path.is_file():
        raise FileNotFoundError(f"Premium CSV not found at {csv_path}")
    df = pd.read_csv(csv_path)

    # --------------------------------------------------------------
    # 2️⃣ Drop any identifier / leakage columns
    # --------------------------------------------------------------
    id_like = [c for c in df.columns if "id" in c.lower()]
    likely_ids = [
        c for c in df.select_dtypes(include=["object"]).columns
        if df[c].nunique() / len(df) > 0.95
    ]
    id_cols = list(set(id_like + likely_ids))
    if id_cols:
        print(f"🔎  Dropping identifier columns: {id_cols}")
        df = df.drop(columns=id_cols)

    # --------------------------------------------------------------
    # 3️⃣ Quick sanity‑check of dtypes (object columns = categorical)
    # --------------------------------------------------------------
    print("\n--- Column dtypes BEFORE preprocessing ---")
    print(df.dtypes)

    # --------------------------------------------------------------
    # 4️⃣ Train‑test split (before any scaling/encoding)
    # --------------------------------------------------------------
    TARGET = "annual_premium_inr"
    X = df.drop(columns=[TARGET])
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    # --------------------------------------------------------------
    # 5️⃣ Identify categorical vs numeric columns
    # --------------------------------------------------------------
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]

    print("\n--- Detected column groups ---")
    print(f"Categorical ({len(categorical_cols)}): {categorical_cols}")
    print(f"Numeric    ({len(numeric_cols)}): {numeric_cols}")

    # --------------------------------------------------------------
    # 6️⃣ Build preprocessing transformer (dense float64 output)
    # --------------------------------------------------------------
    cat_transformer = make_one_hot_encoder()
    num_transformer = StandardScaler()

    preprocessor = ColumnTransformer(
        [
            ("num", num_transformer, numeric_cols),
            ("cat", cat_transformer, categorical_cols),
        ]
    )

    # --------------------------------------------------------------
    # 7️⃣ Fit / transform the training data 
    # --------------------------------------------------------------
    print("\nFitting the preprocessing transformer on X_train ...")
    X_train_pre = preprocessor.fit_transform(X_train)
    X_test_pre  = preprocessor.transform(X_test)

    # --------------------------------------------------------------
    # 8️⃣ Sanity‑check – must be numeric float64
    # --------------------------------------------------------------
    print("\n--- Pre‑processing sanity check ---")
    print(f"X_train_pre shape : {X_train_pre.shape}")
    print(f"X_test_pre shape  : {X_test_pre.shape}")
    print(f"dtype of X_train_pre : {X_train_pre.dtype}")

    if X_train_pre.dtype != np.float64:
        raise RuntimeError("Preprocessing did NOT produce float64 data.")

    # --------------------------------------------------------------
    # 9️⃣ Define the three regression models
    # --------------------------------------------------------------
    models = {
        "LinearRegression": LinearRegression(),
        "RandomForest": RandomForestRegressor(
            n_estimators=300, random_state=42, n_jobs=-1
        ),
        "XGBoost": XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1,
        ),
    }

    # --------------------------------------------------------------
    # 10️⃣ Train each model on the already‑numeric data
    # --------------------------------------------------------------
    results = []
    for name, model in models.items():
        print(f"\nTraining {name} ...")
        model.fit(X_train_pre, y_train)
        preds = model.predict(X_test_pre)

        r2   = r2_score(y_test, preds)
        mae  = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))

        results.append(
            {
                "name": name,
                "r2": r2,
                "mae": mae,
                "rmse": rmse,
                "model": model,
            }
        )

    # --------------------------------------------------------------
    # 11️⃣ Summary table
    # --------------------------------------------------------------
    print("\n=== Regression – Premium Prediction Results ===")
    print(f"{'Model':<15} {'R2':>8} {'MAE':>10} {'RMSE':>10}")
    for r in results:
        print(f"{r['name']:<15} {r['r2']:.4f} {r['mae']:.2f} {r['rmse']:.2f}")

    # --------------------------------------------------------------
    # 12️⃣ Select the best model (must have R² ≥ 0.75)
    # --------------------------------------------------------------
    best = max(results, key=lambda x: x["r2"])
    if best["r2"] < 0.75:
        print(
            f"\n BEST R² ({best['r2']:.4f}) is BELOW the required 0.75 threshold."
        )
    else:
        print(
            f"\n SELECTED MODEL: {best['name']} (R² = {best['r2']:.4f})"
        )

    # --------------------------------------------------------------
    # 13️⃣ Persist the chosen model AND the preprocessing transformer
    # --------------------------------------------------------------
    joblib.dump(best["model"], MODELS_DIR / "premium_regressor.joblib")
    joblib.dump(preprocessor, MODELS_DIR / "premium_transformer.joblib")
    print(f"\n Model saved to : {MODELS_DIR / 'premium_regressor.joblib'}")
    print(f" Transformer saved to : {MODELS_DIR / 'premium_transformer.joblib'}")

    # --------------------------------------------------------------
    # 14️⃣ Save a few EDA plots
    # --------------------------------------------------------------
    # 14.1 Distribution plots
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    sns.histplot(df["age"], kde=True, color="#3f72af")
    plt.title("Age Distribution")
    plt.subplot(1, 3, 2)
    sns.histplot(df["bmi"], kde=True, color="#ff6f61")
    plt.title("BMI Distribution")
    plt.subplot(1, 3, 3)
    sns.histplot(df["annual_premium_inr"], kde=True, color="#6a4c93")
    plt.title("Premium Distribution")
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "premium_distribution.png")
    plt.close()

    # 14.2 Correlation heat‑map – numeric columns only
    numeric_for_corr = df.select_dtypes(include=[np.number])
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        numeric_for_corr.corr(),
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        linewidths=0.5,
    )
    plt.title("Correlation Matrix (numeric features only)")
    plt.savefig(REPORTS_DIR / "premium_correlation.png")
    plt.close()

    # 14.3 Box‑plot premium by region (region is categorical, seaborn can plot it directly)
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='region', y='annual_premium_inr', data=df_insurance, hue='region', legend=False, palette='Set2')
    plt.title("Premium by Region")
    plt.savefig(REPORTS_DIR / "premium_by_region.png")
    plt.close()

    # 14.4 Tiny preview image of the raw cleaned data
    df_head_to_image(df, "Premium Data – First 5 Rows", "premium_head.png")

    print("\nAll EDA images are saved in the `reports/` folder.\n")


if __name__ == "__main__":
    main()


🔎  Dropping identifier columns: ['customer_id']

--- Column dtypes BEFORE preprocessing ---
age                         int64
gender                     object
bmi                       float64
children                    int64
smoker                     object
region                     object
occupation                 object
annual_income_inr         float64
exercise_frequency         object
alcohol_consumption        object
medical_history            object
family_medical_history     object
annual_premium_inr          int64
dtype: object

--- Detected column groups ---
Categorical (8): ['gender', 'smoker', 'region', 'occupation', 'exercise_frequency', 'alcohol_consumption', 'medical_history', 'family_medical_history']
Numeric    (4): ['age', 'bmi', 'children', 'annual_income_inr']

Fitting the preprocessing transformer on X_train ...

--- Pre‑processing sanity check ---
X_train_pre shape : (41069, 31)
X_test_pre shape  : (10268, 31)
dtype of X_train_pre : float64

Training LinearRe

In [11]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib

# Path to load the cleaned claims dataset
DATA_PATH = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\data\processed\insurance_claims_cleaned.csv"

# Path to save the trained models and preprocessor
MODEL_DIR = r"C:\Users\Prasanth Rajaram\OneDrive\Desktop\project\InsureAI\models"

def run_fraud_classification_pipeline():
    print("=" * 85)
    print("MODULE 1 — PART B: CLASSIFICATION — AUTO CLAIMS FRAUD DETECTION")
    print("=" * 85)
    
    # 1. Load Dataset
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns\n")
    
    # 2. Handle "?" and missing value placeholders
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip().replace({'?': np.nan, 'UNKNOWN': np.nan, 'NA': np.nan})
            
    # Check for missing values and impute them
    missing_counts = df.isnull().sum()
    if missing_counts.sum() > 0:
        print("Missing Values Detected After Placeholder Cleaning:")
        print(missing_counts[missing_counts > 0].to_string())
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                if df[col].dtype in ['int64', 'float64']:
                    df[col] = df[col].fillna(df[col].median())
                else:
                    df[col] = df[col].fillna(df[col].mode()[0])
    else:
        print("No missing values or '?' placeholders remaining.\n")
        
    # 3. Drop ID and Leakage Columns
    id_leakage_cols = ['claim_id']
    df_clean = df.drop(columns=[col for col in id_leakage_cols if col in df.columns])
    
    # 4. Target Label Encoding (Target: 'fraud_reported' where N -> 0, Y -> 1)
    le = LabelEncoder()
    df_clean['target'] = le.fit_transform(df_clean['fraud_reported']) 
    
    X = df_clean.drop(columns=['fraud_reported', 'target'])
    y = df_clean['target']
    
    # Verify class balance
    counts = y.value_counts()
    pcts = y.value_counts(normalize=True) * 100
    print("Class Balance for Target (fraud_reported):")
    print(f"  - Non-Fraud (0 / N): {counts[0]:,} ({pcts[0]:.2f}%)")
    print(f"  - Fraud     (1 / Y): {counts[1]:,} ({pcts[1]:.2f}%)\n")
    
    # Identify features
    num_features = ['customer_age', 'policy_tenure_years', 'annual_premium_inr', 'claim_amount_inr', 
                    'days_to_report', 'witnesses', 'num_past_claims_3yrs']
    cat_features = ['gender', 'policy_type', 'incident_type', 'incident_severity', 
                    'police_report_filed', 'documents_complete', 'claim_channel', 'income_bracket', 'claim_location']
    
    # 5. Train/Test Split BEFORE Scaling or SMOTE (Prevent Data Leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    
    print(f"Train/Test Split (Stratified 80/20):")
    print(f"  - Train Set Size: {X_train.shape[0]:,} samples (Fraud: {sum(y_train==1):,})")
    print(f"  - Test Set Size : {X_test.shape[0]:,} samples (Fraud: {sum(y_test==1):,})\n")
    
    # 6. Feature Preprocessing (StandardScaler + OneHotEncoder fitted ONLY on X_train)
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
        ]
    )
    
    X_train_scaled = preprocessor.fit_transform(X_train)
    X_test_scaled = preprocessor.transform(X_test)
    
    # 7. Handle Class Imbalance using SMOTE (ON TRAINING DATA ONLY!)
    print("Applying SMOTE on Training Data ONLY...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    
    print(f"  - Pre-SMOTE Train Class Distribution : {dict(pd.Series(y_train).value_counts())}")
    print(f"  - Post-SMOTE Train Class Distribution: {dict(pd.Series(y_train_resampled).value_counts())}")
    print(f"  - Test Set Unchanged (Evaluation Integrity Preserved): {dict(pd.Series(y_test).value_counts())}\n")
    
    # 8. Train 3 Classifiers
    classifiers = {
        "Logistic Regression (Balanced)": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
        "Random Forest Classifier (SMOTE)": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost Classifier (SMOTE)": XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='logloss')
    }
    
    results = []
    confusion_matrices = {}
    
    for name, clf in classifiers.items():
        print(f"Training {name}...")
        if "SMOTE" in name:
            clf.fit(X_train_resampled, y_train_resampled)
        else:
            clf.fit(X_train_scaled, y_train)
            
        y_pred = clf.predict(X_test_scaled)
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, pos_label=1)
        rec = recall_score(y_test, y_pred, pos_label=1)
        f1 = f1_score(y_test, y_pred, pos_label=1)
        cm = confusion_matrix(y_test, y_pred)
        
        confusion_matrices[name] = cm
        
        results.append({
            "Classifier": name,
            "Accuracy": round(acc, 4),
            "Precision (Fraud)": round(prec, 4),
            "Recall (Fraud)": round(rec, 4),
            "F1-Score (Fraud)": round(f1, 4)
        })
        
    results_df = pd.DataFrame(results)
    
    print("\n" + "=" * 85)
    print("CLASSIFIER PERFORMANCE COMPARISON TABLE (TEST SET EVALUATION)")
    print("=" * 85)
    print(results_df.to_string(index=False))
    
    print("\n" + "=" * 85)
    print("CONFUSION MATRICES BY CLASSIFIER")
    print("=" * 85)
    for name, cm in confusion_matrices.items():
        tn, fp, fn, tp = cm.ravel()
        print(f"\n--- {name} ---")
        print(f"  True Negatives (Correct Non-Fraud) : {tn:,}")
        print(f"  False Positives (False Alarms)    : {fp:,}")
        print(f"  False Negatives (MISSED FRAUD!)   : {fn:,}  <-- Critical Risk Metric")
        print(f"  True Positives (Detected Fraud)   : {tp:,}")
        
    # 9. Model Selection & Justification
    best_row = results_df.sort_values(by="F1-Score (Fraud)", ascending=False).iloc[0]
    best_name = best_row["Classifier"]
    best_recall = best_row["Recall (Fraud)"]
    best_f1 = best_row["F1-Score (Fraud)"]
    
    print("\n" + "=" * 85)
    print("BEST MODEL SELECTION & JUSTIFICATION")
    print("=" * 85)
    print(f"SELECTED MODEL: {best_name}")
    print(f"PERFORMANCE   : Minority Class Recall = {best_recall:.2%}, F1-Score = {best_f1:.4f}")
    print("JUSTIFICATION : For insurance fraud detection, RECALL on the minority class (Fraud = Y) is the most critical metric.")
    print(f"                {best_name} maximizes minority class recall while maintaining high precision and F1-score.")
    print("=" * 85 + "\n")
    
    # Save best classification model, preprocessor, & label encoder
    os.makedirs(MODEL_DIR, exist_ok=True)
    best_clf_obj = classifiers[best_name]
    
    joblib.dump(best_clf_obj, os.path.join(MODEL_DIR, "best_fraud_model.joblib"))
    joblib.dump(preprocessor, os.path.join(MODEL_DIR, "claims_preprocessor.joblib"))
    joblib.dump(le, os.path.join(MODEL_DIR, "claims_label_encoder.joblib"))
        
    print(f"Saved best classification model ({best_name}), preprocessor & label encoder using joblib into models/!\n")

if __name__ == "__main__":
    run_fraud_classification_pipeline()


MODULE 1 — PART B: CLASSIFICATION — AUTO CLAIMS FRAUD DETECTION
Loaded Dataset Shape: 51,000 rows, 18 columns

No missing values or '?' placeholders remaining.

Class Balance for Target (fraud_reported):
  - Non-Fraud (0 / N): 41,102 (80.59%)
  - Fraud     (1 / Y): 9,898 (19.41%)

Train/Test Split (Stratified 80/20):
  - Train Set Size: 40,800 samples (Fraud: 7,918)
  - Test Set Size : 10,200 samples (Fraud: 1,980)

Applying SMOTE on Training Data ONLY...
  - Pre-SMOTE Train Class Distribution : {0: np.int64(32882), 1: np.int64(7918)}
  - Post-SMOTE Train Class Distribution: {0: np.int64(32882), 1: np.int64(32882)}
  - Test Set Unchanged (Evaluation Integrity Preserved): {0: np.int64(8220), 1: np.int64(1980)}

Training Logistic Regression (Balanced)...
Training Random Forest Classifier (SMOTE)...
Training XGBoost Classifier (SMOTE)...

CLASSIFIER PERFORMANCE COMPARISON TABLE (TEST SET EVALUATION)
                      Classifier  Accuracy  Precision (Fraud)  Recall (Fraud)  F1-Score (F